   # 1.0 Data Cleaning and Staging
   ---
   ---
   
   In this notebook we extract the raw Kaggle transactional data for "Online Retail II UCI", remove PII (if there is any), and stage the data for feature engineering. Information on the data set is available on the following path:
   https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data

   It is an OLTP type information set for 2 years worth of orders.

   ##### Please set up the local environment per the below:
```
📂 rfm-lifecycle-segmentation/
│
├── 📂 data/
│   ├── 📂 interim/       <- Staging tables etc
│   ├── 📂 processed/     <- Output data sets/ presentation layer
│   └── 📂 raw/           <- Untouched orders.csv, customers.csv, etc
│
├── 📂 models             <- Trained models, predictions, summaries etc
│
├── 📂 notebooks/         <- Jupyter notebooks location.
│   └── 📄 1_data_clean_and_stage.ipynb
│
├── 📂 reports/           <- Graphics, Dashboards, Visuals etc
│   └── 📂 figures/       <- eg. rfm_view.png
│
├── 📂 src/               <- Source code
│   └── 📄 functions.py   <- Python functions
│
├── 📄 .gitignore         <- Ignore data/, .venv/, .env etc
│
├── 📄 requirements.txt   <- pandas, scikit-learn, seaborn, etc
│
└── 📄 README.md          <- Executive summary
```

---
## ETL Contents
| **Stage** | **Step / Field** | **Description** |
| :--- | :--- | :--- |
| **Extract Data** | | Fetch raw data from Kaggle API |
| **Transform Data** | | Cleaning and formatting |
| | 0. Data Preparation | Remove leading/trailing edges, make uppercase, etc |
| | 1. Invoice (Code) | Validate invoice codes |
| | 1b. Cancelled | Create cancellation flag |
| | 2. Stock Code | Validate stock codes |
| | 3. Description | Validate descriptions|
| | 4. Quantity | Validate quantities |
| | 5. Invoice Date | Convert Invoice Date to datetime format |
| | 6. Price | Create UnitPrice and TotalPrice fields|
| | 7. Country | Format country |
| | 8. Customer ID | Pseudonymise customer ID and format |
| | 9. ETL Validation | Validate transformed data prior load |
| **Load Data** | | Export to staging directory (interim) |
| | Table Definitions | Table specifications for stg_online_retail_orders data set |

---
# Extract Data
---
---

To download data via Kaggle's API we need to install kagglehub. For reference I already installed {pandas, numpy, matplotlib, seaborn, scikit-learn, ipykernel, ipywidgets, pyarrow} in the virtual (.venv) environment. Required packages are listed in requirements.txt

**We download the file to cache (and then clear):**

In [ ]:
import pandas as pd
import kagglehub
import shutil
import os

# Define the input path for the source data

input_csv_path = "../data/raw/online_retail_II.csv"
input_dir = os.path.dirname(input_csv_path)

# 1. Check if the required directory structure exists
if not os.path.exists(input_dir):
    print(f"❌ ERROR: The directory '{input_dir}' does not exist.")
    print("Action Required - Please set up your local environment:")
    print("  1. Create the standard data folders: 'data/raw/', 'data/interim/', and 'data/processed/'.")
    print("  2. Ensure 'data/' is added to your .gitignore file.")
    print("Once complete, rerun this cell.")
    
# 2. If directory exists, check if the file already exists
elif not os.path.exists(input_csv_path):
    
    # =============================
    # Download the file from Kaggle
    # =============================
    
    print("Downloading dataset from Kaggle...")
    cache_path = kagglehub.dataset_download("mashlyn/online-retail-ii-uci")
    print(f"✅ Downloaded to cache: {cache_path}")

    # ============================================
    # Import downloaded file into df_raw dataframe
    # ============================================

    # Locate .csv file in the cache folder
    files = os.listdir(cache_path)
    csv_filename =[f for f in files if f.endswith('.csv')][0]
    source_file = os.path.join(cache_path, csv_filename)
	
    # Define destination
    destination_file = input_csv_path
    
    # Import data into data frame df_raw
    print(f"Importing online_retail_II data from {destination_file}..")
    shutil.copy(source_file, destination_file)
    df_raw = pd.read_csv(destination_file)  
    print("✅ Import complete.")
    
    # =======================================
    # Locate folder and clear the cached file
    # =======================================
        
    dataset_cache_dir = os.path.dirname(os.path.dirname(cache_path)) 
    shutil.rmtree(dataset_cache_dir)
    print("✅ Cache cleared.")

# 3. If file already exists in raw, load but advise to review.
else:
    print(f"❌ File '{input_csv_path}' already exists.")
    print("  Stopping the Kaggle download.")
    print("  Reloading existing online_retail_II.csv into dataframe df_raw ..")
    df_raw = pd.read_csv(input_csv_path)
    print("✅ Import to dataframe df_raw complete.")
    print("   Please review existing online_retail_II.csv file for completeness.")
    print("   Delete existing file if it is incomplete and rerun this cell.")

100%|██████████| 14.5M/14.5M [00:01<00:00, 9.12MB/s]

Extracting files...


✅ Downloaded to cache: /Users/sachaplaye/.cache/kagglehub/datasets/mashlyn/online-retail-ii-uci/versions/3
Importing online_retail_II data from ../data/raw/online_retail_II.csv..
✅ Import complete.
✅ Cache cleared.


**We check field names, datatypes, counts and nulls prior transform steps:**

In [2]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


**Check a sample of 5 lines:**

In [3]:
# Pivot the below ON/Off to view the raw data for exploratory data analysis.
# df_raw.sample(5)

# For PII, we redact customer ID when posting to notebooks on GitHub.
display_df = df_raw.sample(5).copy()

# Apply the redaction and display
display_df['Customer ID'] = '[REDACTED]'
display(display_df)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
130970,501847,22423,REGENCY CAKESTAND 3 TIER,2,2010-03-21 10:30:00,12.75,[REDACTED],United Kingdom
262377,514705,22059,CERAMIC STRWBERRY DESIGN MUG,12,2010-07-05 15:21:00,1.49,[REDACTED],United Kingdom
458271,532811,79160,HEART SHAPE WIRELESS DOORBELL,48,2010-11-14 15:27:00,1.69,[REDACTED],United Kingdom
887751,568417,84032A,CHARLIE+LOLA PINK HOT WATER BOTTLE,12,2011-09-27 11:10:00,2.95,[REDACTED],EIRE
755348,557112,22960,JAM MAKING SET WITH JARS,1,2011-06-16 16:31:00,8.29,[REDACTED],United Kingdom


---
# Transform data
---
---

**We re-link the data frame (df_raw) to source so we can run the cleanse section independently:**

In [1]:
import pandas as pd
import os

# Define the input path for the raw data
input_csv_path = "../data/raw/online_retail_II.csv"
input_dir = os.path.dirname(input_csv_path)

# 1. Check if the required directory structure exists
if not os.path.exists(input_dir):
    print(f"❌ ERROR: The directory '{input_dir}' does not exist.")
    print("Action Required - Please set up your local environment:")
    print("  1. Create the standard data folders: 'data/raw/', 'data/interim/', and 'data/processed/'.")
    print("  2. Ensure 'data/' is added to your .gitignore file.")
    print("Once complete, run Data Extract steps first then rerun this cell.")

# 2. If directory exists, check if the file already exists
elif not os.path.exists(input_csv_path):
    print(f"❌ ERROR: The file '{input_csv_path}' does not exist.")
    print("Action Required - Please run Data Extract steps first then rerun this cell.")

# 3. If file exists, import
else:
    print("Importing online_retail_II.csv to dataframe df_raw ..")
    df_raw = pd.read_csv(input_csv_path)
    print("✅ Import to dataframe df_raw complete.")

Importing online_retail_II.csv to dataframe df_raw ..
✅ Import to dataframe df_raw complete.


---
## The transform steps:
Here we are only looking at transforming the data set to make it clean for loading in to the data/interim layer. *(IE datatype, standardise format, remove spaces and such like)*

### The reference material on Kaggle:
| **Field Name** | **Description** |
| :--- | :--- |
| **Invoice** | Invoice number. Nominal. A 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter 'c', it indicates a cancellation. | 
| **StockCode** | Product (item) code. Nominal. A 5-digit integral number uniquely assigned to each distinct product. | 
| **Description** | Product (item) name. Nominal. | 
| **Quantity** | The quantities of each product (item) per transaction. Numeric. | 
| **InvoiceDate** | Invoice date and time. Numeric. The day and time when a transaction was generated. | 
| **UnitPrice** | Unit price. Numeric. Product price per unit in sterling (Â£). | 
| **CustomerID** | Customer number. Nominal. A 5-digit integral number uniquely assigned to each customer. | 
| **Country** | Country name. Nominal. The name of the country where a customer resides. | 

*(I don't add data types above since they don't all match the descriptions)*

**What we check:**
- Invoice: Since it is stored as string we need to check it is only 6-digit with some including c - cancellation. We can add boolean true/false field for Cancellation field.
- StockCode: I can see one is "84596B" so we need to check this.
- Description: We leave untouched but can remove leading and trailing edges.
- Quantity: This already came through as int64, we can check for -ve but would leave untouched.  
- InvoiceDate: This is string remove leading and trailing edges check for date consistency and convert to datetime.
- Price is already float64 so again we check but would leave untouched.
- Country: We need to check what countries reomve leading and trailing edges and standardise any obvious typos that are clearly meant to be a country like France etc.
- We enforce data types to ensure Quantity is integer, Price is float, and Invoice, StockCode, Description, and Country are strings.
- Prior to the load we will pseudonymise the 'Customer ID' for use downstream in /data/interim/ although this is usually handled by services like AWS KMS (Key Management Service), Azure Key Vault or similar.

---

### 0. We make string fields uppercase, handle wildcards and remove leading/trailing edges:

In [2]:
# Create working copy first, so we can always refer back to the original raw data
df_clean = df_raw.copy()

# ===============================================================
#    == Apply blanket cleaning rules to standardise strings ==
# ===============================================================

print("Processing data ..")

# Standardise Invoice field:
df_clean['Invoice'] = (
    df_clean['Invoice']
    .str.upper()                                      # 1. Make uppercase
    .str.replace(r'[^A-Z0-9\s]', ' ', regex=True)     # 2. Replace symbols with space
    .str.replace(r'\s+', ' ', regex=True)             # 3. Collapse multiple spaces into one
    .str.strip()                                      # 4. Remove spaces at the very start/end
)

# tandardise StockCode field:
df_clean['StockCode'] = (
    df_clean['StockCode']
    .str.upper()                                      # 1. Make uppercase
    .str.replace(r'[^A-Z0-9\s]', ' ', regex=True)     # 2. Replace symbols with space
    .str.replace(r'\s+', ' ', regex=True)             # 3. Collapse multiple spaces into one
    .str.strip()                                      # 4. Remove spaces at the very start/end
)

# tandardise Description field:
df_clean['Description'] = (
    df_clean['Description']
    .str.upper()                                      # 1. Make uppercase
    .str.replace(r'[^A-Z0-9\s]', ' ', regex=True)     # 2. Replace symbols with space
    .str.replace(r'\s+', ' ', regex=True)             # 3. Collapse multiple spaces into one
    .str.strip()                                      # 4. Remove spaces at the very start/end
)

# tandardise Country field:
df_clean['Country'] = (
    df_clean['Country']
    .str.upper()                                      # 1. Make uppercase
    .str.replace(r'[^A-Z0-9\s]', ' ', regex=True)     # 2. Replace symbols with space
    .str.replace(r'\s+', ' ', regex=True)             # 3. Collapse multiple spaces into one
    .str.strip()                                      # 4. Remove spaces at the very start/end
)

print("✅ 'Invoice', 'StockCode', 'Description' and 'Country' fields standardised.")

Processing data ..
✅ 'Invoice', 'StockCode', 'Description' and 'Country' fields standardised.


---

### 1. Validate the Invoice field for cancellations:

In [3]:
invoice_col = 'Invoice'  # set input to invoice

s = df_clean['Invoice'].astype(str)

counts_by_length = s.str.len().value_counts().sort_index()
has_c = s.str.contains('C', case=False, na=False)
c_at_start = s.str.startswith('C')
c_at_end = s.str.endswith('C')
other_chars = s.str.contains(r'[^A-Z0-9]', na=False)

print("Counts by length:")
print(counts_by_length)

print("\nTotal rows:", len(s))
print("Rows with C anywhere:", has_c.sum())
print("Rows with C at start:", c_at_start.sum())
print("Rows with C at end:", c_at_end.sum())
print("Rows with other non-alphanumeric chars:", other_chars.sum())

# Sample some cancellations
problem_rows = df_clean[has_c | other_chars]
problem_rows[[invoice_col]].drop_duplicates().head(5)

Counts by length:
Invoice
6    1047871
7      19500
Name: count, dtype: int64

Total rows: 1067371
Rows with C anywhere: 19494
Rows with C at start: 19494
Rows with C at end: 0
Rows with other non-alphanumeric chars: 0


,Invoice
178,C489449
196,C489459
285,C489476
316,C489503
318,C489504


### 1.(b) Create Cancellation field (boolean):

In [4]:
# 1. Create the Cancellation boolean flag based on whether the Invoice starts with 'C'
df_clean['Cancellation'] = df_clean['Invoice'].str.startswith('C')

# 2. Check a few cancelled orders to ensure it works as expected and redact Customer ID
display_df = df_clean[df_clean['Cancellation'] == True].sample(5).copy()
display_df['Customer ID'] = '[REDACTED]'

print("✅ Cancellation field created.")
print("   See sample below:")
display(display_df)

✅ Cancellation field created.
   See sample below:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Cancellation
184235,C506908,79323W,WHITE CHERRY LIGHTS,-1,2010-05-05 10:40:00,6.75,[REDACTED],UNITED KINGDOM,True
647820,C546865,22697,GREEN REGENCY TEACUP AND SAUCER,-1,2011-03-17 15:46:00,2.95,[REDACTED],UNITED KINGDOM,True
790296,C560123,21936,RED RETROSPOT PICNIC BAG,-2,2011-07-15 10:35:00,2.95,[REDACTED],UNITED KINGDOM,True
859924,C566252,22856,ASSORTED EASTER DECORATIONS BELLS,-19,2011-09-11 12:14:00,1.25,[REDACTED],UNITED KINGDOM,True
632993,C545457,22780,LIGHT GARLAND BUTTERFILES PINK,-2,2011-03-02 17:18:00,4.25,[REDACTED],UNITED KINGDOM,True


Note: in the analysis we had 19,500 '7 varchar long' Invoice Code line items with cancellation flags for 19,494 of them. We are missing 6 that are not cancellation entries. We will pick this up in the data exploration phase as cancellations are explicitly stated in the Kaggle documentation but we don't want to set additional more nuanced "data interpretation" rules in the ETL phase without steer from the business.

*Also it is evident from the above data set, we have negative quantities and positive price. Per the documentation from kaggle it looks like Price should be UnitPrice and that we need to add TotalPrice = Quantity x UnitPrice for downstream use of the data. (eg sum (TotalPrice) by Customer will automatically remove cancelled orders from RFM). We will check the distribution of Price later to confirm.*

---

### 2. Validate StockCode for integers/characters:
*(In the data samples above we saw a B at the end so let's check for characters).*

In [5]:
stock = df_clean['StockCode'].astype(str).str.strip()

length_counts = stock.str.len().value_counts().sort_index()
ending_letters = stock.str.extract(r'([A-Z])$')[0]
ending_counts = ending_letters.value_counts().reindex(list('ABCDEFGHIJKLMNOPQRSTUVWXYZ'), fill_value=0)

starts_with_letter = stock.str.contains(r'^[A-Z]', na=False).sum()
has_letter_middle = stock.str.contains(r'^[0-9]+[A-Z]+[0-9]+', na=False).sum()

print("StockCode counts by length:")
print(length_counts)

print("\nStockCode counts ending with A-Z:")
print(ending_counts)

print("\nStockCodes starting with a letter:", starts_with_letter)
print("StockCodes with letters in the middle:", has_letter_middle)

# optional: inspect examples
print("\nExamples starting with a letter:")
print(stock[stock.str.contains(r'^[A-Z]', na=False)].drop_duplicates().head(10).tolist())

print("\nExamples with letters in the middle:")
print(stock[stock.str.contains(r'^[0-9]+[A-Z]+[0-9]+', na=False)].drop_duplicates().head(10).tolist())

StockCode counts by length:
StockCode
1       1713
2        283
3       1446
4       2158
5     932385
6     127591
7       1392
8        127
9         74
12       202
Name: count, dtype: int64

StockCode counts ending with A-Z:
0
A    32464
B    36538
C    16412
D     9747
E     5438
F     4728
G     3649
H      644
I       42
J      569
K      946
L     6503
M     2442
N     1646
O       19
P     2332
Q        0
R      316
S     4551
T     3674
U      393
V       93
W     1276
X        0
Y       36
Z       19
Name: count, dtype: int64

StockCodes starting with a letter: 6093
StockCodes with letters in the middle: 0

Examples starting with a letter:
['POST', 'D', 'DCGS0058', 'DCGS0068', 'DOT', 'M', 'DCGS0004', 'DCGS0076', 'C2', 'BANK CHARGES']

Examples with letters in the middle:
[]


*Checking the above there aren't just stock code line items but things like 'Bank Charges' (IE administrative and operational codes). We note this for later and leave StockCode unchanged.*

---

### 3. Description (field) we leave this untouched, it is already Uppercase and trimmed.

Normally this would be part of the MDM - Master Data Management along with 'Country' (that should reference the ISO-3166 Country Codes). Description returned 1062989 non-null. We leave missing values and typos to be handled further downstream.

---
### 4. Quantity: We leave this as integer.

**Quality check we look at the distribution of -ve quantities versus cancellations:**

In [6]:
# Count of negative and positive quantities by Cancellation
neg_pos_counts = df_clean.groupby('Cancellation').agg(
    Negative_Quantity=('Quantity', lambda x: (x < 0).sum()),
    Positive_Quantity=('Quantity', lambda x: (x > 0).sum())
)

print(neg_pos_counts)

              Negative_Quantity  Positive_Quantity
Cancellation                                      
False                      3457            1044420
True                      19493                  1


It looks like there is a positive quantity versus cancellation and 3.5k negative quantities for no cancellation. 

**Let's check this:**

In [7]:
# Sample 5 full rows where Cancellation is False and Quantity is negative
sample_rows = df_clean[(df_clean['Cancellation'] == False) & (df_clean['Quantity'] < 0)].sample(5)
sample_rows

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Cancellation
673569,549172,21637,NaN,-139,2011-04-06 17:27:00,0.0,NaN,UNITED KINGDOM,False
436986,531165,16015,NaN,-270,2010-11-05 14:28:00,0.0,NaN,UNITED KINGDOM,False
83556,497174,20900,NaN,-11,2010-02-05 17:25:00,0.0,NaN,UNITED KINGDOM,False
82904,497001,82609,NaN,-9,2010-02-05 11:41:00,0.0,NaN,UNITED KINGDOM,False
145781,503239,46000P,NaN,-10,2010-03-30 17:52:00,0.0,NaN,UNITED KINGDOM,False


In [8]:
# Get Invoice number for cancelled line item with positive quantity
cancelled_positive_invoice = df_clean[(df_clean['Cancellation'] == True) & (df_clean['Quantity'] > 0)]['Invoice']

# Get the row for that Invoice number
row_invoice = df_clean[df_clean['Invoice'].isin(cancelled_positive_invoice)]

#Display the row
row_invoice

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Cancellation
76799,C496350,M,MANUAL,1,2010-02-01 08:24:00,373.57,NaN,UNITED KINGDOM,True


*We can comment the findings in the documentation but leave the data unchanged.*

---

### 5. InvoiceDate we convert to datetime.
*We saw from df_raw.info(), there are no NaT's*

**We check they are all in 'YYYY-MM-DD HH:MM:SS' format to get no datetime conversion fails:**

In [9]:
invoice_dates = df_clean['InvoiceDate'].astype(str)
pattern = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$'

valid_mask = invoice_dates.str.match(pattern, na=False)

print("✅ All rows match YYYY-MM-DD HH:MM:SS is", valid_mask.all())
print("Invalid rows count:", (~valid_mask).sum())

# inspect any bad values
df_clean.loc[~valid_mask, 'InvoiceDate'].head(10)

✅ All rows match YYYY-MM-DD HH:MM:SS is True
Invalid rows count: 0


Series([], Name: InvoiceDate, dtype: str)

**We convert InvoiceDate to datetime:**

In [10]:
# The format parameter using YYYY-MM-DD HH:MM:SS format
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], format='%Y-%m-%d %H:%M:%S')
print("✅ Converted InvoiceDate to datetime64")

✅ Converted InvoiceDate to datetime64


In [11]:
# Validation: Check for nulls in InvoiceDate before and after conversion
raw_missing = df_raw['InvoiceDate'].isna().sum()
clean_missing = df_clean['InvoiceDate'].isna().sum()

if raw_missing == clean_missing:
    print("✅ Validation passed: No data was lost during datetime conversion.")
else:
    print(f"❌ Warning: Data loss detected. Had {raw_missing} nulls, now has {clean_missing}.")

✅ Validation passed: No data was lost during datetime conversion.


---
### 6. Price may need changing to UnitPrice and TotalPrice = Quantity x UnitPrice 

Let's test this first by looking at the distribution of prices. Max price, Min Price and then distribute by buckets including zero price.

In [12]:
# Split the min/max range into 10 equal buckets and count them
print(df_clean['Price'].value_counts(bins=10).sort_index())

(-53686.924999999996, -44337.924]          1
(-44337.924, -35081.488]                   2
(-35081.488, -25825.052]                   0
(-25825.052, -16568.616]                   0
(-16568.616, -7312.18]                     2
(-7312.18, 1944.256]                 1067225
(1944.256, 11200.692]                    125
(11200.692, 20457.128]                    13
(20457.128, 29713.564]                     2
(29713.564, 38970.0]                       1
Name: count, dtype: int64


*Here we see large negative numbers like (£53,686.92) may be outliers, corrections or similar. Nearly all the Prices are under £1,945 and the Description, Quantity and Price listed in the samples table make it clear these are Unit Prices, which is also backed by the table specification in Kaggle*

**We Change Price to UnitPrice and Introduce TotalPrice to assist downstream analysis:**


In [13]:
df_clean = df_clean.rename(columns={'Price': 'UnitPrice'})
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']
print("✅ Renamed Price to UnitPrice")
print("✅ Created TotalPrice column")

✅ Renamed Price to UnitPrice
✅ Created TotalPrice column


---

### 7. We Leave "Customer Id" for now and look at Country

We need to check all the spelling types of the country field to know what to standardise. As mentioned previously this should be part of MDM reference but we will manually code the standard adjustment for this exercise.

In [14]:
# Look at all unique countries to spot typos
print(df_clean['Country'].value_counts().sort_index())

Country
AUSTRALIA                 1913
AUSTRIA                    938
BAHRAIN                    126
BELGIUM                   3123
BERMUDA                     34
BRAZIL                      94
CANADA                     228
CHANNEL ISLANDS           1664
CYPRUS                    1176
CZECH REPUBLIC              30
DENMARK                    817
EIRE                     17866
EUROPEAN COMMUNITY          61
FINLAND                   1049
FRANCE                   14330
GERMANY                  17624
GREECE                     663
HONG KONG                  364
ICELAND                    253
ISRAEL                     371
ITALY                     1534
JAPAN                      582
KOREA                       63
LEBANON                     58
LITHUANIA                  189
MALTA                      299
NETHERLANDS               5140
NIGERIA                     32
NORWAY                    1455
POLAND                     535
PORTUGAL                  2620
RSA                        169


**There are no country name typos but let's standardise and keep to the same format**

The Country field contains abbreviations. I won't use ISO 3166 country table to standardise since Channel Islands, European Community won't map to a country code.

**So we clean up the abbreviations to match the long form:**

In [15]:
# 1. Change to long name format
country_rename = {
    'EIRE': 'IRELAND', 
    'RSA': 'SOUTH AFRICA',
    'USA': 'UNITED STATES'
}
df_clean['Country'] = df_clean['Country'].replace(country_rename)
print("✅ Country names standardized.")

✅ Country names standardized.


---

### 8. We pseudonymise 'Customer ID' *(before pushing the dataset into data/interim).*

1. Naming Convention: rename 'Customer ID' to 'CustomerID'.
2. Convert the data type to integer-string removing the .0 decimal place.
3. The analysis shows there are 5942 unique customer numbers. We use HMAC-SHA256 to pseudonymise.

    *Normally we use services like AWS KMS or Azure Key Vault for this, (also we don't store the key locally in this exercise).*

**1. We rename Customer ID field:**

In [16]:
# 1. Rename to standardise Field naming convention
df_clean = df_clean.rename(columns={'Customer ID': 'CustomerID'})
print("✅ Customer ID header standardised to CustomerID.")

✅ Customer ID header standardised to CustomerID.


**Before proceeding to step 2 let's check the CustomerID:**

In [17]:
# Count CustomerID metrics in one summary table
cust = df_clean['CustomerID']

total_rows = len(cust)
null_count = cust.isna().sum()
non_null_count = total_rows - null_count

cust_str = cust.dropna().astype(str).str.strip()

length_summary = (
    cust_str
    .str.len()
    .value_counts()
    .sort_index()
    .rename_axis('CustomerID_Length')
    .reset_index(name='Count')
)

dot_zero_count = cust_str.str.endswith('.0').sum()
unique_non_null = cust_str.nunique()

summary_rows = pd.DataFrame([
    {'CustomerID_Length': 'Total rows', 'Count': total_rows},
    {'CustomerID_Length': 'Non-null CustomerID', 'Count': non_null_count},
    {'CustomerID_Length': 'Null CustomerID', 'Count': null_count},
    {'CustomerID_Length': 'CustomerID ends with .0', 'Count': dot_zero_count},
    {'CustomerID_Length': 'Unique CustomerID (non-null)', 'Count': unique_non_null},
])

summary = pd.concat([length_summary, summary_rows], ignore_index=True, sort=False)
display(summary)

,CustomerID_Length,Count
0,7,824364
1,Total rows,1067371
2,Non-null CustomerID,824364
3,Null CustomerID,243007
4,CustomerID ends with .0,824364
5,Unique CustomerID (non-null),5942


Per the table it looks like all customer ID's are 5 digits long with .0 at the end.

**2. We convert the data type to integer-string with no .0 at the end:** 

In [18]:
# 2. Convert to Int, then to String
# Create function to handle logic
def clean_id(val):
    if pd.isna(val):
        return val           # If it's a NaN, leave it alone
    return str(int(val))     # If it's a number, convert float -> int -> string

# Apply it to the CustomerID field
df_clean['CustomerID'] = df_clean['CustomerID'].apply(clean_id)

print("✅ CustomerID formatted to integer-string.")

✅ CustomerID formatted to integer-string.


**3. We pseudonymise to CustomerHashID and drop CustomerID from df_load for stage 3 of ETL:**

In [19]:

# import libraries for hashing
import hmac
import hashlib
import secrets
import numpy as np

# Generate a random 64-character hex string (32 bytes)
SECRET_KEY = secrets.token_hex(32).encode('utf-8')

def generate_customer_hash(customer_id_str):
    # 1. Check if the CustomerID is null, missing, or literal string "nan"
    if pd.isna(customer_id_str) or str(customer_id_str).strip().lower() in ['nan', 'none', '']:
        return np.nan  # Return a true null, do NOT hash it
    
    # 2. Otherwise, hash the valid ID
    return hmac.new(SECRET_KEY, str(customer_id_str).encode('utf-8'), hashlib.sha256).hexdigest()

# Apply the HMAC-SHA256 function to df_clean
df_clean['CustomerHashID'] = df_clean['CustomerID'].apply(generate_customer_hash)

# Select fields for loading (and drop 'CustomerID')
final_selection =[
    'Invoice', 
    'StockCode', 
    'Description', 
    'Quantity', 
    'InvoiceDate', 
    'UnitPrice', 
    'TotalPrice', 
    'CustomerHashID', 
    'Country', 
    'Cancellation'
]

# Transition from clean to load, applying the final selection of fields
df_load = df_clean[final_selection]

print("✅ Hash applied and df_load created.")

✅ Hash applied and df_load created.


---

### Pre-load Validations and Tests
**We do some data checks on df_load prior to /data/interim export:**

In [20]:
# Check column names, data types, and non-null counts
df_load.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 10 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   Invoice         1067371 non-null  str           
 1   StockCode       1067371 non-null  str           
 2   Description     1062989 non-null  str           
 3   Quantity        1067371 non-null  int64         
 4   InvoiceDate     1067371 non-null  datetime64[us]
 5   UnitPrice       1067371 non-null  float64       
 6   TotalPrice      1067371 non-null  float64       
 7   CustomerHashID  824364 non-null   str           
 8   Country         1067371 non-null  str           
 9   Cancellation    1067371 non-null  bool          
dtypes: bool(1), datetime64[us](1), float64(2), int64(1), str(5)
memory usage: 176.9 MB


In [21]:
# Green screen test, sample 5 rows to visually check the data set versus the original
df_load.sample(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,TotalPrice,CustomerHashID,Country,Cancellation
835713,564172,22138,BAKING SET 9 PIECE RETROSPOT,4,2011-08-23 14:19:00,4.95,19.80,bd4c3a6adc4b46ada461f9d887ac3740a4aa7dd61e8f12...,UNITED KINGDOM,False
436418,531043,22628,PICNIC BOXES SET OF 3 RETROSPOT,1,2010-11-05 12:00:00,4.95,4.95,4742a57e9b58ab969faec773a0568476771c5d691130fa...,UNITED KINGDOM,False
211135,509850,21410,COUNTRY COTTAGE DOORSTOP GREEN,3,2010-05-26 09:00:00,4.25,12.75,fd343a2ca48ad738e636d32dca9e66313f43f15b5701c3...,FRANCE,False
633938,C545537,22456,NATURAL SLATE CHALKBOARD LARGE,-1,2011-03-03 14:14:00,4.95,-4.95,bf6df489913450572378bfaa4420ba2d4da7e1f1610240...,UNITED KINGDOM,True
35558,492368,85232B,SET 3 RUSSIAN DOLL STACKING TINS,2,2009-12-16 14:08:00,4.95,9.90,0630627a835725913f5426762f58fd33fb17508ac22f32...,UNITED KINGDOM,False


In [22]:
# Additional explicit tests on row count, cell matching and customer counts.

# 1. Define which fields to check 
columns_to_check =[
    'Invoice', 
    'StockCode', 
    'Description', 
    'Quantity', 
    'InvoiceDate', 
    'UnitPrice', 
    'TotalPrice', 
    'Country', 
    'Cancellation'
]

# 2. Row Count Check
assert len(df_clean) == len(df_load), f"Row count mismatch! Clean: {len(df_clean)}, Load: {len(df_load)}"
print(f"Row counts match: {len(df_load):,} rows")

# 3. use equals() to checkif values, data types, and row order are identical
data_matches = df_clean[columns_to_check].equals(df_load[columns_to_check])

if data_matches:
    print("✅ Data integrity confirmed: All underlying data match.")
else:
    print("❌ Warning: Data mismatch detected in non-ID columns.")

# 4. Hash Mapping Check that we still have exactly the same number of unique customers
original_unique = df_clean['CustomerID'].nunique()
hashed_unique = df_load['CustomerHashID'].nunique()

print(f"Original Unique IDs: {original_unique}")
print(f"Hashed Unique IDs: {hashed_unique}")

assert original_unique == hashed_unique, "❌ Hash collision or missing IDs detected."
print(f"✅ Pseudonymisation mapping is 1-to-1 ({original_unique} unique customers preserved).")

print("-" * 55)
print("✅ Additional ETL validation checks passed. Ready for Load.")

Row counts match: 1,067,371 rows
✅ Data integrity confirmed: All underlying data match.
Original Unique IDs: 5942
Hashed Unique IDs: 5942
✅ Pseudonymisation mapping is 1-to-1 (5942 unique customers preserved).
-------------------------------------------------------
✅ Additional ETL validation checks passed. Ready for Load.


---
# Load data
---
---

**We push df_load to folder data/interim, file stg_online_retail_orders.parquet:**

In [23]:
import os
# We use parquet format to preserve datatypes in the output file
# run pip install pyarrow in terminal if you don't have it

# Define the output path for the transformed data
output_parquet_path = '../data/interim/stg_online_retail_orders.parquet'
output_dir = os.path.dirname(output_parquet_path)

# 1. Check if the required directory structure exists
if not os.path.exists(output_dir):
    print(f"❌ ERROR: The directory '{output_dir}' does not exist.")
    print("Action Required - Please set up your local environment:")
    print("  1. Create the standard data folders: 'data/raw/', 'data/interim/', and 'data/processed/'.")
    print("  2. Ensure 'data/' is added to your .gitignore file.")
    print("Once complete, rerun this cell.")

# 2. If directory exists, check if the file already exists
elif not os.path.exists(output_parquet_path):
    print(f"Exporting online retail orders data to {output_parquet_path} ..")
    df_load.to_parquet(output_parquet_path, index=False)
    print("✅ Export complete.")

# 3. If file already exists, skip the export and advise to review.
else:
    print(f"❌ File '{output_parquet_path}' already exists.")
    print("  .. Skipping export.")
    print("  Please review/delete existing file, prior to re-running this cell.")

Exporting online retail orders data to ../data/interim/stg_online_retail_orders.parquet ..
✅ Export complete.


---
# Table Specifications
---
---

#### **Table name:** stg_online_retail_orders
- Fields: 10
- Rows: 1.067,371
- Approximate Memory Use: 74.3 MB

| **Field** | **Description** | **Data Type** | **Data Type Description** | **Example** |
| :--- | :--- | :--- | :--- | :--- |
| Invoice | This is the invoice number | String | Number or C+Number for cancellations | 516171 |
| StockCode | This is the product code| String | Number and additional admin/ops type letters | 22487 |
| Description | Full product description | String | String of concatenated key words | PLASTERS IN TIN SPACEBOY |
| Quantity | number of products measure | Integer 64 | There are negative numbers here for adjustments | 3 |
| InvoiceDate | Date of Purchase / order | Datetime 64[us] | This is datetime YYYY-MM-DD hh:mm:ss form| 2011-11-21 12:06:00 |
| UnitPrice | Price of one line item | Float 64 | These are Pound Sterling and all positive | 0.39 |
| TotalPrice | Total price of lines items | Float 64 | This is Quantity x UnitPrice in Pound Sterling | 1.17 |
| CustomerHashID | Pseudonymised customer number | String | Simulated hash using HMAC-SHA256 | 09c72970956ea... |
| Country | Full country names | String | Country names including regions/unspecified | UNITED KINGDOM |
| Cancellation | Flag for line item changes | Boolean | This is a True/False flag | False |

---
---